<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/03_Governance_Security_and_Data_Quality.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F03_Governance_Security_and_Data_Quality.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/03_Governance_Security_and_Data_Quality.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/03_Governance_Security_and_Data_Quality.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/03_Governance_Security_and_Data_Quality.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>
<br clear="all"/>

---

# Track 1 (Notebook 03): Dataplex Universal Catalog, Lineage & BNM RMiT / PDPA Fine-Grained Security
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ Notebook 03 Architecture & Governance Flow (`acsm_bronze` → `acsm_silver` → `acsm_gold`)

![Track 1 Notebook 03 — Dataplex Universal Catalog, Lineage & Fine-Grained Security Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook3_governance_security_flow.png)

---

### 📋 Notebook 03 Step-by-Step Execution Summary
1. **Step 1 (Setup & APIs)**: Auto-detect active `PROJECT_ID`, `USER_EMAIL`, and Singapore region (`asia-southeast1`), and enable the Dataplex Universal Catalog, Data Lineage, Data Catalog Policy Tag, and BigQuery Data Policy APIs.
2. **Step 2 (`%%bigquery` SQL — 100% Column Descriptions on Silver & Gold)**: Enrich all **79 columns** across the 4 `acsm_silver` tables and `acsm_gold.gold_aeon360_customer_profile` with business glossary descriptions (`C1.1.1.10`) and audit **100% metadata coverage** across `acsm_bronze`, `acsm_silver`, and `acsm_gold`.
3. **Step 3 (`%%bigquery` SQL — Dataplex Governance Labels & Facets)**: Attach standardized **BNM RMiT & Malaysian PDPA governance labels** (`data_domain`, `medallion_layer`, `bnm_rmit_tier`, `pdpa_contains_pii`) to all datasets and tables (`C1.1.1.8`).
4. **Step 4 (Table & Column-Level Data Lineage)**: Audit the end-to-end lineage graph from `acsm_bronze` (including the GCP Lakehouse Iceberg view `m3CIF` and AWS Glue Federated Iceberg view `dimProduct`) $\rightarrow$ `acsm_silver` $\rightarrow$ `acsm_gold.gold_aeon360_customer_profile` (`C1.1.1.9`), with click-by-click BigQuery Studio UI verification instructions.
5. **Step 5 (Column-Level Dynamic Data Masking via Dataplex Policy Tags — `CLS`)**: Create a real Dataplex Policy Tag Taxonomy (`ACSM_BNM_RMiT_PDPA_Classification`), bind **SHA-256 Masking** to `CIF_NM` (Customer Name) and **Default Null/Zero Masking** to `B_NetIncome` (Monthly Net Income), attach the Policy Tags to `acsm_gold.gold_aeon360_customer_profile`, and run a live SQL query to see masked columns in action (`C1.1.1.24`, `C1.1.5.3`).
6. **Step 6 (Row-Level Security by Malaysian State via `CREATE ROW ACCESS POLICY` — `RLS`)**: Apply a native BigQuery Row Access Policy restricting your active user session to **Central Region Malaysian States (`Selangor`, `Kuala Lumpur`, `Putrajaya`, `Negeri Sembilan`)** and verify that queries simultaneously enforce **both** Row-Level Filtering and Column-Level Dynamic Masking (`C1.1.1.24`, `C1.1.5.5`).
7. **Step 7 (1-Click Unmask & Policy Reset for Tracks 2–4)**: Cleanly remove the temporary workshop restriction on `acsm_gold.gold_aeon360_customer_profile` and verify that all **100,000 customer rows** (across all 16 Malaysian states) and unmasked columns are restored for **Track 2 (BQML)**, **Track 3 (BQCA)**, and **Track 4 (Google ADK Agents)**.

---
## Step 1: Configure Parameters (`PROJECT_ID`, `USER_EMAIL` & Singapore Region) and Enable Governance APIs
This cell auto-detects your active Google Cloud `PROJECT_ID` and authenticated `USER_EMAIL`, sets `LOCATION = "asia-southeast1"` (Singapore), and enables the required Google Cloud Governance & Security APIs:
* `dataplex.googleapis.com` — **Dataplex Universal Catalog**
* `datalineage.googleapis.com` — **Automated Table & Column-Level Data Lineage**
* `datacatalog.googleapis.com` — **Policy Tag Taxonomies for Column-Level Security**
* `bigquerydatapolicy.googleapis.com` — **BigQuery Dynamic Data Masking Policies (`SHA256` / `DEFAULT_MASKING_VALUE`)**


In [ ]:
# @title Step 1: Auto-Detect `PROJECT_ID` & `USER_EMAIL` and Enable Dataplex, Lineage & Data Policy APIs
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

USER_EMAIL = subprocess.check_output(
    ["gcloud", "config", "get-value", "account"], text=True
).strip()

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["USER_EMAIL"] = USER_EMAIL

!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable   dataplex.googleapis.com   datalineage.googleapis.com   datacatalog.googleapis.com   bigquerydatapolicy.googleapis.com   bigquery.googleapis.com   --project={PROJECT_ID} --quiet

print("==========================================================================")
print(f"✅ Active GCP Project ID : {PROJECT_ID}")
print(f"👤 Active Workshop User  : {USER_EMAIL}")
print(f"🌏 Governed Region       : {LOCATION} (Singapore)")
print("==========================================================================")


---
## Step 2: Populate & Audit 100% Column-Level Business Descriptions Across `acsm_silver` & `acsm_gold` (`C1.1.1.10`)
In **Notebook 1**, we populated business glossary descriptions from `Mock Metadata.xlsx` across our `acsm_bronze` tables. When **Notebook 2** created the 4 `acsm_silver` tables and `acsm_gold.gold_aeon360_customer_profile` via `CREATE OR REPLACE TABLE ... AS SELECT`, the new Silver and Gold columns were created without column-level `OPTIONS(description=...)`.

### Step 2.1 — Enrich All 79 Columns in `acsm_silver` and `acsm_gold` with Governed Business Descriptions
This BigQuery SQL cell attaches authoritative ACSM business descriptions to every column across all 4 Silver tables (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`) and the Gold AEON 360 Feature Store (`gold_aeon360_customer_profile`) so Dataplex Universal Catalog, BigQuery Conversational Analytics (BQCA), and AI agents have 100% semantic context.


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 2.1 (Pure BigQuery SQL): Populate 100% Column-Level Business Descriptions
-- across all 4 `acsm_silver` tables and `acsm_gold.gold_aeon360_customer_profile`
-- =============================================================================

-- 1. Enrich `acsm_silver.silver_customer_cif` (18 columns)
ALTER TABLE `acsm_silver.silver_customer_cif`
  ALTER COLUMN CIF_ID SET OPTIONS (description = 'Unique ACSM Customer ID deduplicated from m3CIF [Primary Key]'),
  ALTER COLUMN record_refresh_date SET OPTIONS (description = 'Latest customer master extraction date parsed from Rcd_DT'),
  ALTER COLUMN CIF_NM SET OPTIONS (description = 'Customer full legal name [Malaysian PDPA Sensitive PII - Subject to SHA256 Dynamic Masking]'),
  ALTER COLUMN Gender SET OPTIONS (description = 'Customer gender code (M = Male, F = Female)'),
  ALTER COLUMN MaritalSts SET OPTIONS (description = 'Customer marital status (SINGLE, MARRIED, WIDOWED, DIVORCED)'),
  ALTER COLUMN Citizen SET OPTIONS (description = 'Customer citizenship classification (MALAYSIAN / NON-MALAYSIAN)'),
  ALTER COLUMN State SET OPTIONS (description = 'Malaysian State of residence [Row-Level Security Filter Key]'),
  ALTER COLUMN Region SET OPTIONS (description = 'Malaysian macro-region (Central, Northern, Southern, East Coast, East Malaysia)'),
  ALTER COLUMN Race SET OPTIONS (description = 'Customer race classification [BNM Fair-Lending Protected Attribute]'),
  ALTER COLUMN Occupation SET OPTIONS (description = 'Customer declared occupation category'),
  ALTER COLUMN EmpSts SET OPTIONS (description = 'Customer employment status code'),
  ALTER COLUMN N_Age SET OPTIONS (description = 'Customer age in years'),
  ALTER COLUMN N_YrStay SET OPTIONS (description = 'Years residing at current home address'),
  ALTER COLUMN N_YrJob SET OPTIONS (description = 'Years employed at current employer'),
  ALTER COLUMN B_NetIncome SET OPTIONS (description = 'Verified monthly net income in MYR [Confidential Financial PII - Subject to Dynamic Masking]'),
  ALTER COLUMN B_GrossIncome SET OPTIONS (description = 'Verified monthly gross income in MYR'),
  ALTER COLUMN B_AnnualIncome SET OPTIONS (description = 'Verified annual gross income in MYR'),
  ALTER COLUMN RecvPromo_FG SET OPTIONS (description = 'Malaysian PDPA Section 43 marketing & cross-sell opt-in consent flag (Y / N)');

-- 2. Enrich `acsm_silver.silver_ep_underwriting` (18 columns)
ALTER TABLE `acsm_silver.silver_ep_underwriting`
  ALTER COLUMN APPL_NO SET OPTIONS (description = 'Easy Payment (EP) product application number'),
  ALTER COLUMN AGREE_NO SET OPTIONS (description = 'Easy Payment loan agreement account number'),
  ALTER COLUMN CIF_ID SET OPTIONS (description = 'Unique ACSM Customer ID linked to silver_customer_cif'),
  ALTER COLUMN application_date SET OPTIONS (description = 'Easy Payment application date parsed from APPL_DT'),
  ALTER COLUMN APPL_STS SET OPTIONS (description = 'Easy Payment application status (Approved / Rejected / Pending)'),
  ALTER COLUMN SCORING_POINT SET OPTIONS (description = 'Easy Payment internal credit scoring points'),
  ALTER COLUMN SCORING_RANK SET OPTIONS (description = 'Easy Payment credit score risk rank bucket'),
  ALTER COLUMN SCORE_DECISION SET OPTIONS (description = 'Automated credit engine decision (Accept / Decline)'),
  ALTER COLUMN LOAN_GRP SET OPTIONS (description = 'Easy Payment financing product group code'),
  ALTER COLUMN FIN_AMT SET OPTIONS (description = 'Approved Easy Payment principal financed amount in MYR [Reconciliation Control Metric]'),
  ALTER COLUMN INST_AMT SET OPTIONS (description = 'Monthly Easy Payment installment amount in MYR'),
  ALTER COLUMN INTEREST SET OPTIONS (description = 'Easy Payment effective interest / profit rate'),
  ALTER COLUMN TOTAL_INST SET OPTIONS (description = 'Total installment tenure in months'),
  ALTER COLUMN NetIncome SET OPTIONS (description = 'Applicant monthly net income at application time in MYR'),
  ALTER COLUMN NDI SET OPTIONS (description = 'Applicant Net Disposable Income (NDI) in MYR'),
  ALTER COLUMN CUR_DSR SET OPTIONS (description = 'Applicant pre-loan Debt Service Ratio (DSR %)'),
  ALTER COLUMN NEW_DSR SET OPTIONS (description = 'Applicant post-loan Debt Service Ratio (DSR %)'),
  ALTER COLUMN TOTAL_AEON_OSB SET OPTIONS (description = 'Total existing AEON outstanding balance across all products in MYR');

-- 3. Enrich `acsm_silver.silver_cc_underwriting` (16 columns)
ALTER TABLE `acsm_silver.silver_cc_underwriting`
  ALTER COLUMN Appl_ID SET OPTIONS (description = 'Credit Card application ID'),
  ALTER COLUMN Account_No SET OPTIONS (description = 'Credit Card account number'),
  ALTER COLUMN CIF_ID SET OPTIONS (description = 'Unique ACSM Customer ID linked to silver_customer_cif'),
  ALTER COLUMN application_date SET OPTIONS (description = 'Credit Card application date parsed from Appl_DT'),
  ALTER COLUMN ApplSts_ID SET OPTIONS (description = 'Credit Card application decision status (Approve / Reject)'),
  ALTER COLUMN CardTyp_ID SET OPTIONS (description = 'Credit Card tier type (Classic, Gold, Platinum, Infinite)'),
  ALTER COLUMN CardBrand_ID SET OPTIONS (description = 'Credit Card network brand (Visa, Master, JCB)'),
  ALTER COLUMN ScoreDecision_ID SET OPTIONS (description = 'Credit Card underwriting score decision code'),
  ALTER COLUMN ScoreRank_ID SET OPTIONS (description = 'Credit Card underwriting score rank grade'),
  ALTER COLUMN NetIncome SET OPTIONS (description = 'Applicant monthly net income at CC application in MYR'),
  ALTER COLUMN NDI SET OPTIONS (description = 'Applicant Net Disposable Income (NDI) at CC application in MYR'),
  ALTER COLUMN CurrDSR SET OPTIONS (description = 'Applicant current Debt Service Ratio (DSR %)'),
  ALTER COLUMN NewDSR SET OPTIONS (description = 'Applicant post-card Debt Service Ratio (DSR %)'),
  ALTER COLUMN B_CrLimit SET OPTIONS (description = 'Approved Credit Card limit in MYR [Reconciliation Control Metric]'),
  ALTER COLUMN Final_Score SET OPTIONS (description = 'External CTOS bureau credit score'),
  ALTER COLUMN Final_ScoreDesc SET OPTIONS (description = 'External CTOS bureau score band description');

-- 4. Enrich `acsm_silver.silver_collections_summary` (5 columns)
ALTER TABLE `acsm_silver.silver_collections_summary`
  ALTER COLUMN CIF_ID SET OPTIONS (description = 'Unique ACSM Customer ID across EP and CC collection ledgers'),
  ALTER COLUMN total_ep_unpaid_osp SET OPTIONS (description = 'Total unpaid overdue principal across Easy Payment contracts in MYR'),
  ALTER COLUMN total_cc_unpaid_osp SET OPTIONS (description = 'Total unpaid overdue principal across Credit Card accounts in MYR'),
  ALTER COLUMN combined_unpaid_osp SET OPTIONS (description = 'Combined EP + CC unpaid overdue principal in MYR [Reconciliation Control Metric]'),
  ALTER COLUMN worst_collection_score_grade SET OPTIONS (description = 'Worst delinquency collection score grade across EP and CC portfolios');

-- 5. Enrich `acsm_gold.gold_aeon360_customer_profile` (22 columns)
ALTER TABLE `acsm_gold.gold_aeon360_customer_profile`
  ALTER COLUMN CIF_ID SET OPTIONS (description = 'Unique ACSM Customer ID [Primary Entity Key]'),
  ALTER COLUMN CIF_NM SET OPTIONS (description = 'Customer full legal name [Malaysian PDPA Sensitive PII - Protected by SHA256 Dynamic Masking Policy Tag]'),
  ALTER COLUMN State SET OPTIONS (description = 'Malaysian State of residence [Row-Level Security Filter Key]'),
  ALTER COLUMN Region SET OPTIONS (description = 'Malaysian geographical region'),
  ALTER COLUMN Occupation SET OPTIONS (description = 'Customer occupation category'),
  ALTER COLUMN N_Age SET OPTIONS (description = 'Customer age in years'),
  ALTER COLUMN B_NetIncome SET OPTIONS (description = 'Verified monthly net income in MYR [Confidential Financial PII - Protected by Default Null Dynamic Masking Policy Tag]'),
  ALTER COLUMN B_AnnualIncome SET OPTIONS (description = 'Verified annual gross income in MYR'),
  ALTER COLUMN RecvPromo_FG SET OPTIONS (description = 'Malaysian PDPA marketing consent flag (Y = Opted-In, N = Opted-Out)'),
  ALTER COLUMN ep_app_count SET OPTIONS (description = 'Total lifetime Easy Payment (EP) applications submitted'),
  ALTER COLUMN total_ep_financed_myr SET OPTIONS (description = 'Total cumulative Easy Payment principal financed in MYR'),
  ALTER COLUMN avg_ep_new_dsr SET OPTIONS (description = 'Average post-loan Debt Service Ratio (DSR %) across EP applications'),
  ALTER COLUMN cc_app_count SET OPTIONS (description = 'Total lifetime Credit Card (CC) applications submitted'),
  ALTER COLUMN total_cc_limit_myr SET OPTIONS (description = 'Total approved Credit Card limit across all cards in MYR'),
  ALTER COLUMN latest_ctos_score SET OPTIONS (description = 'Latest CTOS credit bureau score from Credit Card underwriting'),
  ALTER COLUMN total_ep_unpaid_osp SET OPTIONS (description = 'Total unpaid overdue principal on Easy Payment accounts in MYR'),
  ALTER COLUMN total_cc_unpaid_osp SET OPTIONS (description = 'Total unpaid overdue principal on Credit Card accounts in MYR'),
  ALTER COLUMN combined_unpaid_osp SET OPTIONS (description = 'Total combined unpaid overdue principal (EP + CC) in MYR'),
  ALTER COLUMN worst_collection_score_grade SET OPTIONS (description = 'Worst collection score grade across EP and CC accounts'),
  ALTER COLUMN active_card_count SET OPTIONS (description = 'Count of active credit cards federated from AWS Glue dimProduct table'),
  ALTER COLUMN total_cp_usage_myr SET OPTIONS (description = 'Total Credit Purchase utilization in MYR federated from AWS Glue dimProduct table'),
  ALTER COLUMN total_cp_available_myr SET OPTIONS (description = 'Total remaining Credit Purchase available limit in MYR federated from AWS Glue dimProduct table');


### Step 2.2 — Audit Metadata Coverage Across All Three Medallion Layers (`acsm_bronze`, `acsm_silver`, `acsm_gold`)
Run this query to verify via `INFORMATION_SCHEMA.COLUMN_FIELD_PATHS` that **100% of columns** across `acsm_bronze` (6 Native Fact tables), `acsm_silver` (4 Silver tables), and `acsm_gold` (`gold_aeon360_customer_profile`) now carry governed business descriptions in Dataplex Universal Catalog.


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 2.2 (Pure BigQuery SQL): Audit Column Description Coverage Across
-- All 8 `acsm_bronze` Tables & Views, 4 `acsm_silver` Tables, and `acsm_gold`
-- =============================================================================
WITH medallion_catalog AS (
  SELECT '1. Bronze (acsm_bronze)' AS medallion_layer, table_name, column_name, description
  FROM `acsm_bronze.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
  WHERE table_name IN ('Fact_EP_Judge', 'Fact_EP_Sales', 'Fact_EP_Collection', 'Fact_CC_Judge', 'Fact_CC_Sales', 'Fact_CC_Collection', 'm3CIF', 'dimProduct')
  UNION ALL
  SELECT '2. Silver (acsm_silver)' AS medallion_layer, table_name, column_name, description
  FROM `acsm_silver.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
  WHERE table_name IN ('silver_customer_cif', 'silver_ep_underwriting', 'silver_cc_underwriting', 'silver_collections_summary')
  UNION ALL
  SELECT '3. Gold (acsm_gold)' AS medallion_layer, table_name, column_name, description
  FROM `acsm_gold.INFORMATION_SCHEMA.COLUMN_FIELD_PATHS`
  WHERE table_name = 'gold_aeon360_customer_profile'
)
SELECT
  medallion_layer,
  table_name,
  COUNT(*) AS total_columns,
  COUNTIF(description IS NOT NULL AND LENGTH(TRIM(description)) > 0) AS described_columns,
  ROUND(COUNTIF(description IS NOT NULL AND LENGTH(TRIM(description)) > 0) * 100.0 / COUNT(*), 1) AS metadata_coverage_pct
FROM medallion_catalog
GROUP BY 1, 2
ORDER BY 1, 2;

---
## Step 3: Attach Dataplex Governance Labels & Regulatory Facets Across Bronze, Silver, and Gold (`C1.1.1.8`)
To enable CDO data stewards, risk officers, and auditors to filter and discover assets in **Dataplex Universal Catalog Search** by regulatory tier, domain, and PDPA sensitivity, this SQL cell attaches standardized governance labels (`medallion_layer`, `data_domain`, `bnm_rmit_tier`, `pdpa_contains_pii`, `residency`) to our datasets and tables, and audits them via `INFORMATION_SCHEMA.TABLE_OPTIONS`.


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 3 (Pure BigQuery SQL): Attach BNM RMiT & Malaysian PDPA Governance Labels
-- to Datasets and Medallion Tables for Dataplex Universal Catalog Discovery
-- =============================================================================

ALTER SCHEMA `acsm_bronze` SET OPTIONS (
  labels = [('medallion_layer', 'bronze'), ('bnm_rmit_tier', 'tier_1_raw_landing'), ('residency', 'asia_southeast1_sg')]
);
ALTER SCHEMA `acsm_silver` SET OPTIONS (
  labels = [('medallion_layer', 'silver'), ('bnm_rmit_tier', 'tier_1_conformed'), ('pdpa_governed', 'true'), ('residency', 'asia_southeast1_sg')]
);
ALTER SCHEMA `acsm_gold` SET OPTIONS (
  labels = [('medallion_layer', 'gold'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_governed', 'true'), ('residency', 'asia_southeast1_sg')]
);

ALTER TABLE `acsm_silver.silver_customer_cif` SET OPTIONS (
  labels = [('medallion_layer', 'silver'), ('data_domain', 'customer_master'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_contains_pii', 'true')]
);
ALTER TABLE `acsm_silver.silver_ep_underwriting` SET OPTIONS (
  labels = [('medallion_layer', 'silver'), ('data_domain', 'easy_payment_credit'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_contains_pii', 'false')]
);
ALTER TABLE `acsm_silver.silver_cc_underwriting` SET OPTIONS (
  labels = [('medallion_layer', 'silver'), ('data_domain', 'credit_card_underwriting'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_contains_pii', 'false')]
);
ALTER TABLE `acsm_silver.silver_collections_summary` SET OPTIONS (
  labels = [('medallion_layer', 'silver'), ('data_domain', 'collections_recovery'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_contains_pii', 'false')]
);
ALTER TABLE `acsm_gold.gold_aeon360_customer_profile` SET OPTIONS (
  labels = [('medallion_layer', 'gold'), ('data_domain', 'aeon360_customer_risk'), ('bnm_rmit_tier', 'tier_1_critical'), ('pdpa_contains_pii', 'true')]
);

-- Verify attached Dataplex governance labels across Silver & Gold tables
SELECT
  table_schema AS dataset_name,
  table_name,
  option_value AS dataplex_governance_labels
FROM `acsm_silver.INFORMATION_SCHEMA.TABLE_OPTIONS`
WHERE option_name = 'labels'
UNION ALL
SELECT
  table_schema AS dataset_name,
  table_name,
  option_value AS dataplex_governance_labels
FROM `acsm_gold.INFORMATION_SCHEMA.TABLE_OPTIONS`
WHERE option_name = 'labels'
ORDER BY dataset_name, table_name;


---
## Step 4: Verify End-to-End Table & Column-Level Data Lineage (`acsm_bronze` → `acsm_silver` → `acsm_gold` — `C1.1.1.9`)
With `datalineage.googleapis.com` enabled, BigQuery automatically captures both **table-level** and **column-level** lineage whenever `CREATE OR REPLACE TABLE ... AS SELECT` or Dataform pipeline runs execute across `acsm_bronze`, `acsm_silver`, and `acsm_gold`.

Run the cell below to inspect the end-to-end Bronze $ightarrow$ Silver $ightarrow$ Gold lineage map across all 3 storage engines, and then follow the **BigQuery Studio Lineage Tab UI Guide** below it.


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 4 (Pure BigQuery SQL): Inspect End-to-End Medallion Lineage Mapping
-- Across All 3 Storage Engines (`acsm_bronze` -> `acsm_silver` -> `acsm_gold`)
-- =============================================================================
SELECT * FROM UNNEST([
  STRUCT(
    1 AS stage_order,
    'Engine 2: GCP Lakehouse Iceberg (GCS)' AS bronze_storage_engine,
    'acsm_bronze.m3CIF (View -> acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF)' AS bronze_source_object,
    'acsm_silver.silver_customer_cif' AS silver_conformed_table,
    'acsm_gold.gold_aeon360_customer_profile' AS gold_target_table,
    'CIF_ID, CIF_NM, State, Region, Occupation, N_Age, B_NetIncome, B_AnnualIncome, RecvPromo_FG' AS traced_columns
  ),
  STRUCT(
    2,
    'Engine 1: BigQuery Native Storage',
    'acsm_bronze.Fact_EP_Judge',
    'acsm_silver.silver_ep_underwriting',
    'acsm_gold.gold_aeon360_customer_profile',
    'CIF_NO -> CIF_ID, FIN_AMT -> total_ep_financed_myr, NEW_DSR -> avg_ep_new_dsr'
  ),
  STRUCT(
    3,
    'Engine 1: BigQuery Native Storage',
    'acsm_bronze.Fact_CC_Judge',
    'acsm_silver.silver_cc_underwriting',
    'acsm_gold.gold_aeon360_customer_profile',
    'CIF_ID, B_CrLimit -> total_cc_limit_myr, Final_Score -> latest_ctos_score'
  ),
  STRUCT(
    4,
    'Engine 1: BigQuery Native Storage',
    'acsm_bronze.Fact_EP_Collection + acsm_bronze.Fact_CC_Collection',
    'acsm_silver.silver_collections_summary',
    'acsm_gold.gold_aeon360_customer_profile',
    'Unpaid_OSP -> total_ep_unpaid_osp, total_cc_unpaid_osp, combined_unpaid_osp, Score_Grade'
  ),
  STRUCT(
    5,
    'Engine 3: AWS Glue Federated Iceberg (S3)',
    'acsm_bronze.dimProduct (View -> acsm_aws_federated_catalog.acsm_aws_bronze.dimproduct)',
    'Direct Gold Aggregation (card_agg CTE)',
    'acsm_gold.gold_aeon360_customer_profile',
    'Card_Status -> active_card_count, CP_CL_Usage -> total_cp_usage_myr, CP_CL_Available -> total_cp_available_myr'
  )
])
ORDER BY stage_order;


> **🔍 How to Verify Interactive Table & Column-Level Lineage on the GCP Console UI**
> 1. In the **BigQuery Studio Explorer** pane, expand your project $ightarrow$ **`acsm_gold`** $ightarrow$ click **`gold_aeon360_customer_profile`**.
> 2. Click the **Schema** tab to verify that all **22 columns** now display their governed business descriptions from Step 2, and click the **Details** tab to see the **Labels** (`bnm_rmit_tier: tier_1_critical`, `data_domain: aeon360_customer_risk`, `pdpa_contains_pii: true`) from Step 3.
> 3. Click the **Lineage** tab on `gold_aeon360_customer_profile`:
>    * You will see the visual lineage graph linking **`silver_customer_cif`**, **`silver_ep_underwriting`**, **`silver_cc_underwriting`**, **`silver_collections_summary`**, and **`acsm_bronze.dimProduct`** into **`gold_aeon360_customer_profile`**.
>    * Click the **`+`** node on any Silver table (e.g. `silver_customer_cif`) to expand its upstream **`acsm_bronze`** source (`acsm_bronze.m3CIF` $ightarrow$ `acsm_gcp_lakehouse_catalog.acsm_gcp_bronze.m3CIF`).
>    * Click any column name inside the `gold_aeon360_customer_profile` node (such as **`B_NetIncome`** or **`total_ep_financed_myr`**) to highlight its **exact column-level lineage path** from Bronze to Gold!


---
## Step 5: Column-Level Dynamic Data Masking via Dataplex Policy Tags (`CLS` — `C1.1.1.24` & `C1.1.5.3`)
Under **Bank Negara Malaysia (BNM) RMiT Section 10** and the **Malaysian Personal Data Protection Act (PDPA) 2010**, standard BI analysts and data engineers must not view raw customer names (`CIF_NM`) or exact monthly net incomes (`B_NetIncome`) in clear text unless explicitly granted fine-grained clearance—yet their SQL queries and dashboards must continue to run without failing on `Access Denied`.

### Step 5.1 — Provision the Dataplex Policy Tag Taxonomy & Dynamic Data Masking Policies (`SHA256` & `DEFAULT_MASKING_VALUE`)
This Python cell uses the Google Cloud **Data Catalog Policy Tag API** and **BigQuery Data Policy API** in `asia-southeast1` to:
1. Create (or reuse) the taxonomy **`ACSM_BNM_RMiT_PDPA_Classification`** with Fine-Grained Access Control enabled.
2. Create two Policy Tags & attach **Dynamic Data Masking Rules**:
   * **`PII_Customer_Identity_SHA256`** $ightarrow$ Applies **`SHA256`** cryptographic hashing on `STRING` PII columns (`CIF_NM`).
   * **`Confidential_Financial_Income_Null`** $ightarrow$ Applies **`DEFAULT_MASKING_VALUE`** (`0` / `NULL`) on `NUMERIC` financial columns (`B_NetIncome`).
3. Grant your active workshop user (`USER_EMAIL`) the **`roles/bigquerydatapolicy.maskedReader`** (`Fine-Grained Masked Reader`) role on both data policies and attach the Policy Tags to `CIF_NM` and `B_NetIncome` on **`acsm_gold.gold_aeon360_customer_profile`**.


In [ ]:
# @title Step 5.1: Create Dataplex Policy Tags, Bind Dynamic Masking Rules & Attach to `gold_aeon360_customer_profile`
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_session = AuthorizedSession(credentials)
bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
taxonomy_display_name = "ACSM_BNM_RMiT_PDPA_Classification"

# 1. Find or create the Policy Tag Taxonomy in asia-southeast1
tax_url = f"https://datacatalog.googleapis.com/v1/{parent}/taxonomies"
resp = authed_session.get(tax_url)
resp.raise_for_status()
taxonomies = resp.json().get("taxonomies", [])
taxonomy = next((t for t in taxonomies if t.get("displayName") == taxonomy_display_name), None)

if not taxonomy:
    create_resp = authed_session.post(
        tax_url,
        json={
            "displayName": taxonomy_display_name,
            "description": "ACSM BNM RMiT & Malaysian PDPA 2010 Fine-Grained Access Control & Dynamic Data Masking Taxonomy",
            "activatedPolicyTypes": ["FINE_GRAINED_ACCESS_CONTROL"],
        },
    )
    create_resp.raise_for_status()
    taxonomy = create_resp.json()
    print(f"✅ Created Taxonomy: {taxonomy['name']}")
else:
    # Ensure FINE_GRAINED_ACCESS_CONTROL is activated
    if "FINE_GRAINED_ACCESS_CONTROL" not in taxonomy.get("activatedPolicyTypes", []):
        patch_resp = authed_session.patch(
            f"https://datacatalog.googleapis.com/v1/{taxonomy['name']}?updateMask=activatedPolicyTypes",
            json={"activatedPolicyTypes": ["FINE_GRAINED_ACCESS_CONTROL"]},
        )
        patch_resp.raise_for_status()
        taxonomy = patch_resp.json()
    print(f"ℹ️ Using existing Taxonomy: {taxonomy['name']}")

taxonomy_name = taxonomy["name"]

# 2. Find or create the 2 Policy Tags under the Taxonomy
pt_url = f"https://datacatalog.googleapis.com/v1/{taxonomy_name}/policyTags"
pt_list = authed_session.get(pt_url).json().get("policyTags", [])

def get_or_create_policy_tag(display_name, description):
    existing = next((p for p in pt_list if p.get("displayName") == display_name), None)
    if existing:
        return existing["name"]
    r = authed_session.post(pt_url, json={"displayName": display_name, "description": description})
    r.raise_for_status()
    return r.json()["name"]

PT_PII_SHA256 = get_or_create_policy_tag(
    "PII_Customer_Identity_SHA256",
    "Malaysian PDPA Direct Customer Identifier (CIF_NM) — Masked via SHA256 Hash",
)
PT_INCOME_DEFAULT = get_or_create_policy_tag(
    "Confidential_Financial_Income_Null",
    "BNM RMiT Confidential Monthly Net Income (B_NetIncome) — Masked via Default Value (0)",
)
print(f"🏷️ Policy Tag 1 (SHA256 on CIF_NM)        : {PT_PII_SHA256}")
print(f"🏷️ Policy Tag 2 (Default on B_NetIncome)  : {PT_INCOME_DEFAULT}")

# Ensure active user does NOT hold unmasked Fine-Grained Reader on these policy tags during the masking demo
for pt_res in [PT_PII_SHA256, PT_INCOME_DEFAULT]:
    iam_get = authed_session.post(f"https://datacatalog.googleapis.com/v1/{pt_res}:getIamPolicy", json={}).json()
    bindings = [
        b for b in iam_get.get("bindings", [])
        if b.get("role") != "roles/datacatalog.categoryFineGrainedReader"
    ]
    authed_session.post(
        f"https://datacatalog.googleapis.com/v1/{pt_res}:setIamPolicy",
        json={"policy": {"bindings": bindings, "etag": iam_get.get("etag", "")}},
    )

# 3. Create or update the 2 BigQuery Dynamic Data Masking Policies in asia-southeast1
dp_url = f"https://bigquerydatapolicy.googleapis.com/v1/{parent}/dataPolicies"
existing_dps = authed_session.get(dp_url).json().get("dataPolicies", [])

def ensure_masking_policy(policy_id, policy_tag_res, masking_expr):
    existing = next((d for d in existing_dps if d.get("dataPolicyId") == policy_id or d.get("policyTag") == policy_tag_res), None)
    if not existing:
        r = authed_session.post(
            dp_url,
            json={
                "dataPolicyId": policy_id,
                "dataPolicyType": "DATA_MASKING_POLICY",
                "policyTag": policy_tag_res,
                "dataMaskingPolicy": {"predefinedExpression": masking_expr},
            },
        )
        r.raise_for_status()
        dp_name = r.json()["name"]
    else:
        dp_name = existing["name"]

    # Bind roles/bigquerydatapolicy.maskedReader to active workshop user
    iam_resp = authed_session.post(
        f"https://bigquerydatapolicy.googleapis.com/v1/{dp_name}:setIamPolicy",
        json={
            "policy": {
                "bindings": [
                    {
                        "role": "roles/bigquerydatapolicy.maskedReader",
                        "members": [f"user:{USER_EMAIL}"],
                    }
                ]
            }
        },
    )
    iam_resp.raise_for_status()
    return dp_name

dp1 = ensure_masking_policy("acsm_mask_cif_nm_sha256", PT_PII_SHA256, "SHA256")
dp2 = ensure_masking_policy("acsm_mask_net_income_default", PT_INCOME_DEFAULT, "DEFAULT_MASKING_VALUE")
print(f"🛡️ Data Masking Policy 1 (SHA256)         : {dp1}")
print(f"🛡️ Data Masking Policy 2 (DEFAULT_VALUE)  : {dp2}")

# 4. Attach Policy Tags to `CIF_NM` and `B_NetIncome` on `acsm_gold.gold_aeon360_customer_profile`
table_ref = f"{PROJECT_ID}.acsm_gold.gold_aeon360_customer_profile"
table = bq_client.get_table(table_ref)
new_schema = []
for field in table.schema:
    if field.name == "CIF_NM":
        field_dict = field.to_api_repr()
        field_dict["policyTags"] = {"names": [PT_PII_SHA256]}
        new_schema.append(bigquery.SchemaField.from_api_repr(field_dict))
    elif field.name == "B_NetIncome":
        field_dict = field.to_api_repr()
        field_dict["policyTags"] = {"names": [PT_INCOME_DEFAULT]}
        new_schema.append(bigquery.SchemaField.from_api_repr(field_dict))
    else:
        new_schema.append(field)

table.schema = new_schema
bq_client.update_table(table, ["schema"])
print(f"✅ Attached Policy Tags to `CIF_NM` and `B_NetIncome` on `{table_ref}`!")


### Step 5.2 — Verify Live Column-Level Dynamic Data Masking in BigQuery SQL (`%%bigquery`)
Now run a standard `SELECT` query against `acsm_gold.gold_aeon360_customer_profile`. Notice how:
1. The query **succeeds without throwing a permission error**, even though `CIF_NM` and `B_NetIncome` are protected by Policy Tags!
2. **`CIF_NM`** is dynamically masked on-the-fly into a **64-character `SHA256` cryptographic hash**.
3. **`B_NetIncome`** is dynamically masked on-the-fly to **`0`** (`DEFAULT_MASKING_VALUE` for `NUMERIC`), while **`B_AnnualIncome`** (left untagged for side-by-side comparison) and all risk/underwriting metrics remain readable in clear text!


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 5.2 (Pure BigQuery SQL): Query `acsm_gold.gold_aeon360_customer_profile`
-- to Observe Live Column-Level Dynamic Data Masking on `CIF_NM` & `B_NetIncome`
-- =============================================================================
SELECT
  CIF_ID,
  CIF_NM AS cif_nm_masked_sha256,
  State,
  Occupation,
  B_NetIncome AS net_income_masked_default,
  B_AnnualIncome AS annual_income_unmasked_comparison,
  latest_ctos_score,
  active_card_count,
  total_cp_usage_myr
FROM `acsm_gold.gold_aeon360_customer_profile`
ORDER BY total_cp_usage_myr DESC
LIMIT 10;


> **🔍 How to Verify Policy Tags on the GCP Console UI (BigQuery Schema Tab)**
> 1. In the **BigQuery Studio Explorer**, open **`acsm_gold.gold_aeon360_customer_profile`** $ightarrow$ click the **Schema** tab.
> 2. Look at the **Policy tags** column next to **`CIF_NM`** and **`B_NetIncome`**:
>    * `CIF_NM` displays the badge **`ACSM_BNM_RMiT_PDPA_Classification : PII_Customer_Identity_SHA256`** (`Data masking: SHA256`).
>    * `B_NetIncome` displays the badge **`ACSM_BNM_RMiT_PDPA_Classification : Confidential_Financial_Income_Null`** (`Data masking: Default masking value`).


---
## Step 6: Row-Level Security by Malaysian State (`RLS` — `C1.1.1.24` & `C1.1.5.5`)
Next, we demonstrate **Row-Level Security (`CREATE OR REPLACE ROW ACCESS POLICY`)** on `acsm_gold.gold_aeon360_customer_profile`. Suppose your identity represents a **Central Region Branch Manager** who is only authorized to view customers residing in **Central Region Malaysian States (`Selangor`, `Kuala Lumpur`, `Putrajaya`, `Negeri Sembilan`)**.

### Step 6.1 — Create the Row Access Policy Bound to Your Active Workshop User
This cell creates the native BigQuery Row Access Policy `rlp_central_region_branch_manager` on `acsm_gold.gold_aeon360_customer_profile` granted to `user:<YOUR_EMAIL>` with `FILTER USING (State IN ('Selangor', 'Kuala Lumpur', 'Putrajaya', 'Negeri Sembilan'))`.


In [ ]:
# @title Step 6.1: Create Native Row Access Policy (`rlp_central_region_branch_manager`) on `gold_aeon360_customer_profile`
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

rls_ddl = f"""
CREATE OR REPLACE ROW ACCESS POLICY `rlp_central_region_branch_manager`
ON `{PROJECT_ID}.acsm_gold.gold_aeon360_customer_profile`
GRANT TO ('user:{USER_EMAIL}')
FILTER USING (State IN ('Selangor', 'Kuala Lumpur', 'Putrajaya', 'Negeri Sembilan'));
"""

bq_client.query(rls_ddl).result()
print("==========================================================================")
print("✅ Created Row Access Policy: `rlp_central_region_branch_manager`")
print(f"👤 Applied To Principal     : user:{USER_EMAIL}")
print("🗺️ Allowed Malaysian States : Selangor, Kuala Lumpur, Putrajaya, Negeri Sembilan")
print("==========================================================================")


### Step 6.2 — Verify Combined Row-Level Security (RLS) + Column-Level Dynamic Masking (CLS) in SQL (`%%bigquery`)
Run the two queries below against `acsm_gold.gold_aeon360_customer_profile`:
1. **State Distribution Query**: Even though the underlying table holds **100,000 customers across 16 Malaysian states**, BigQuery automatically filters the scan so **only the Central Region states (`Selangor`, `Kuala Lumpur`, `Negeri Sembilan`)** are visible!
2. **Customer-Level Query**: Demonstrates **RLS + CLS working together simultaneously** — only Central Region rows are returned, AND `CIF_NM` (`SHA256`) and `B_NetIncome` (`0`) remain dynamically masked!


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 6.2a (Pure BigQuery SQL): Verify Row-Level Security State Filtering
-- Only Central Region Malaysian States are visible (out of 16 total states)!
-- =============================================================================
SELECT
  State AS visible_malaysian_state,
  Region,
  COUNT(*) AS visible_customer_rows,
  ROUND(SUM(total_ep_financed_myr), 2) AS total_ep_financed_myr,
  ROUND(SUM(total_cp_usage_myr), 2) AS total_card_spend_myr
FROM `acsm_gold.gold_aeon360_customer_profile`
GROUP BY 1, 2
ORDER BY visible_customer_rows DESC;


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 6.2b (Pure BigQuery SQL): Verify Simultaneous Row-Level Security (RLS)
-- + Column-Level Dynamic Data Masking (CLS) on `gold_aeon360_customer_profile`
-- =============================================================================
SELECT
  CIF_ID,
  CIF_NM AS cif_nm_masked_sha256,
  State AS rls_filtered_central_state,
  B_NetIncome AS net_income_masked_default,
  B_AnnualIncome AS annual_income_unmasked,
  latest_ctos_score,
  combined_unpaid_osp
FROM `acsm_gold.gold_aeon360_customer_profile`
ORDER BY combined_unpaid_osp DESC
LIMIT 10;


---
## Step 7: 1-Click Unmask & Policy Reset for Downstream Tracks 2, 3 & 4
Before moving on to **Track 2 (BigQuery ML Credit Delinquency & Cross-Sell Models)**, **Track 3 (BigQuery Conversational Analytics)**, and **Track 4 (Google ADK Multi-Agent System)**, run this final reset cell so your workshop user account has full access to all **100,000 customer rows across all 16 Malaysian states** and unmasked columns on `acsm_gold.gold_aeon360_customer_profile` (while keeping the Dataplex Policy Tag Taxonomy intact in Dataplex Universal Catalog).


In [ ]:
# @title Step 7.1: Drop Workshop Row Access Policy & Detach Column Policy Tags on `gold_aeon360_customer_profile` for Tracks 2–4
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# 1. Drop all row access policies on `acsm_gold.gold_aeon360_customer_profile`
bq_client.query(
    f"DROP ALL ROW ACCESS POLICIES ON `{PROJECT_ID}.acsm_gold.gold_aeon360_customer_profile`;"
).result()
print("✅ Dropped Row Access Policy on `acsm_gold.gold_aeon360_customer_profile` (all 16 Malaysian states restored).")

# 2. Detach Policy Tags from `CIF_NM` and `B_NetIncome` on `gold_aeon360_customer_profile` so Track 2 BQML & Track 3 BQCA read unmasked values immediately
table_ref = f"{PROJECT_ID}.acsm_gold.gold_aeon360_customer_profile"
table = bq_client.get_table(table_ref)
clean_schema = []
for field in table.schema:
    field_dict = field.to_api_repr()
    field_dict.pop("policyTags", None)
    clean_schema.append(bigquery.SchemaField.from_api_repr(field_dict))

table.schema = clean_schema
bq_client.update_table(table, ["schema"])
print("✅ Restored unmasked access on `CIF_NM` and `B_NetIncome` for Track 2 BQML, Track 3 BQCA, and Track 4 ADK Agents!")


In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- Step 7.2 (Pure BigQuery SQL): Confirm Full 100,000 Rows (All 16 States)
-- and Unmasked Columns Are Restored on `acsm_gold.gold_aeon360_customer_profile`
-- =============================================================================
SELECT
  COUNT(*) AS total_gold_customer_rows,
  COUNT(DISTINCT State) AS distinct_malaysian_states,
  ANY_VALUE(CIF_NM) AS sample_unmasked_cif_nm,
  ROUND(AVG(B_NetIncome), 2) AS avg_unmasked_net_income_myr
FROM `acsm_gold.gold_aeon360_customer_profile`;
